In [4]:
import pandas as pd
import os
import re
from mistralai.client import Mistral
from rapidfuzz import fuzz, process
from tqdm import tqdm

In [11]:
client = Mistral(api_key="WevERlQztAvFUIAUFbL2sf2Az39MjZLC")

In [14]:
# Upload the file first
with open("../pdf/party_list_33_6.pdf", "rb") as f:
    uploaded = client.files.upload(
        file={"file_name": "constituency_10_1.pdf", "content": f},
        purpose="ocr"
    )

# Get signed URL
signed_url = client.files.get_signed_url(file_id=uploaded.id)

# Process OCR
result = client.ocr.process(
    model="mistral-ocr-latest",
    document={"type": "document_url", "document_url": signed_url.url}
)

print(result.pages[0].markdown)

|  ขอบเขต
ของบัญชีรายชื่อ
ของสภาค
การเมือง | ชื่อ
พรรคการเมือง | ได้คะแนน
(ให้การถกที่สร้างลานลวดตัวอักษร) | หมายเหตุ  |
| --- | --- | --- | --- |
|  ๑ | ไทยทรัพย์ทวี | ๕๖๘ (ห้าร้อยสามสิบแปด) |   |
|  ๒ | เพื่อชาติไทย | ๓,๓๑๐ (สามพันสามร้อยสิบ) |   |
|  ขอบเขต
ของบัญชีรายชื่อ
ของสภาค
การเมือง | ชื่อ
พรรคการเมือง | ได้คะแนน
(ให้การถกที่สร้างลานลวดตัวอักษร) | หมายเหตุ  |
|  ๓ | ไทม์ | ๒๗๔ (สองร้อยเจ็ดสิบเก้า) |   |
|  ๔ | มีติไทม์ | ๒,๒๑๓ (สองพันสองร้อยสิบสาม) |   |
|  ๕ | รวมใจไทย | ๑๖๔ (สี่ร้อยสามสิบสี่) |   |
|  ๖ | รวมไทยสร้างชาติ | ๘๔๘ (แปดร้อยเก้าสิบแปด) |   |
|  ๗ | พลวัต | ๑๒๖ (หนึ่งร้อยสี่สิบหก) |   |
|  ๘ | ประชาธิปไตยไทม์ | ๗๐๓ (เจ็ดร้อยสาม) |   |
|  ๙ | เพื่อไทย | ๒๓,๓๕๗ (สองหมื่นสามพันสามร้อยห้าสิบเจ็ด) |   |
|  ๑๐ | ทางเลือกไทม์ | ๒๕๘ (สองร้อยห้าสิบแปด) |   |
|  ๑๑ | เศรษฐกิจ | ๒,๒๐๔ (สองพันสองร้อยเก้า) |   |
|  ๑๒ | เสร็จรวมไทย | ๒๒๗ (สองร้อยสี่สิบเจ็ด) |   |
|  ๑๓ | รวมพลังประชาชน | ๓๖๓ (สามร้อยหกสิบสาม) |   |
|  ๑๔ | ท้องที่ไทย | ๕๕ (ห้าสิบห้า) |   |
|  ๑๕ | อนาคตไทย | ๕

In [26]:
md = result.pages[0].markdown

In [5]:
def thai_digit_to_arabic(text):
    thai_digits = {'๐':'0','๑':'1','๒':'2','๓':'3','๔':'4',
                   '๕':'5','๖':'6','๗':'7','๘':'8','๙':'9'}
    return ''.join(thai_digits.get(c, c) for c in text)

In [31]:
res = {}

for line in md.split('\n'):
    # Skip header and separator rows
    if '---' in line or 'พรรคการเมือง' in line or 'รวมคะแนน' in line:
        continue
    
    parts = [p.strip() for p in line.split('|') if p.strip()]
    if len(parts) >= 3:
        party = parts[1].strip()
        vote_raw = parts[2].strip()
        
        # Extract Thai numeral part before the parenthesis
        match = re.match(r'([๐-๙,]+)', vote_raw)
        if match and party:
            vote_str = match.group(1).replace(',', '')
            vote_arabic = thai_digit_to_arabic(vote_str)
            try:
                res[party] = int(vote_arabic)
            except ValueError:
                pass

print(res)

{'ไทยทรัพย์ทวี': 568, 'เพื่อชาติไทย': 3310, 'ไทม์': 274, 'มีติไทม์': 2213, 'รวมใจไทย': 164, 'รวมไทยสร้างชาติ': 848, 'พลวัต': 126, 'ประชาธิปไตยไทม์': 703, 'เพื่อไทย': 23357, 'ทางเลือกไทม์': 258, 'เศรษฐกิจ': 2204, 'เสร็จรวมไทย': 227, 'รวมพลังประชาชน': 363, 'ท้องที่ไทย': 55, 'อนาคตไทย': 58, 'พลังเพื่อไทย': 184, 'ไทยชนะ': 136, 'พลังสังคมไทม์': 33, 'สังคมประชาธิปไตยไทย': 43, 'จิตรชิน': 33, 'ไทรวมพลัง': 125, 'ก้าวอิสระ': 28, 'ปวงชนไทย': 50, 'วิชชั้นไทม์': 43, 'เพื่อชีวิตไทม์': 26, 'คลองไทย': 62, 'ประชาธิปไตย': 1053, 'ไทยก้าวหน้า': 70, 'ไทยภักดี': 185, 'แรงงานสร้างชาติ': 128, 'ประชากรไทย': 342, 'ครูไทยเพื่อประชาชน': 234, 'ประชาชาติ': 255, 'สร้างอนาคตไทย': 111, 'รักชาติ': 96, 'ไทยพร้อม': 181, 'ภูมิใจไทย': 21166, 'พลังธรรมไทม์': 164, 'กรีน': 73, 'ไทยธรรม': 40, 'แผ่นดินธรรม': 30, 'กล้าธรรม': 158, 'พลังประชาธิปไตย': 81, 'โอกาสไทม์': 34, 'เป็นธรรม': 36, 'ประชาชน': 16674, 'ประชาไทย': 107, 'ไทยสร้างไทย': 182, 'ไทยก้าวไทม์': 149, 'ประชากรสชาติ': 6, 'พร้อม': 20, 'เครือข่ายชาวนาแห่งประเทศไทย': 15, 'ไทย

In [6]:
def thai_num_to_int(text: str) -> int:

    thai_to_arabic = str.maketrans("๐๑๒๓๔๕๖๗๘๙", "0123456789")
    text = text.translate(thai_to_arabic)

    # remove non-numeric prefixes
    text = re.sub(r"[^\d,]", " ", text)

    match = re.search(r"[\d,]+", text)
    if not match:
        raise ValueError("No numeric value found")

    return int(match.group().replace(",", ""))

In [7]:
KNOWN_PARTIES = [
    'ประชาธิปัตย์', 'ภูมิใจไทย', 'เศรษฐกิจ', 'กล้าธรรม', 'พลวัต',
    'ประชาชน', 'เพื่อไทย', 'ไทยภักดี', 'รวมไทยสร้างชาติ', 'ปวงชนไทย',
    'ไทยสร้างไทย', 'โอกาสใหม่', 'วิชชั่นใหม่', 'ประชาธิปไตยใหม่',
    'รักชาติ', 'ไทยก้าวใหม่', 'ทางเลือกใหม่', 'พลังประชารัฐ',
    'ประชากรไทย', 'ความหวังใหม่', 'อนาคตไทย', 'บ้านเมือง',
    'เพื่อบ้านเมือง', 'ไทยก้าวหน้า', 'รวมไทยสร้าง', 'แรงงานสร้างชาติ',
    'ไทยชนะ', 'พร้อม', 'ไทยพิทักษ์ธรรม', 'ไทยก้าวไทย', 'ไทยก้าวใหม',
    'เสรีรวมไทย', 'เป็นธรรม', 'ไทยธรรม', 'ฟิวชน', 'ฟิวชัน',
    'รวมพลังประชาชน', 'แรงงานสร้างไทย', 'ไทยสร้างชาติ',
    'ปวงชนชาวไทย', 'กลาธรรม', 'สังคมประชาธิปไตยไทย', 'คลองไทย',
    'รวมใจไทย', 'รวมไทยสร้างชา', 'รักษ์ธรรม', 'ชาติ', 'ไทยพร้อม',
    'สร้างชาติ', 'พลังสังคมใหม่', 'สร้างอนาคตไทย',
    'ไทยทรัพย์ทวี', 'รวมพลัง', 'ไทรวมพลัง', 'เพื่อชาติไทย', 'มิติใหม่',
    'ท้องที่ไทย', 'พลังเพื่อไทย', 'ก้าวอิสระ', 'เพื่อชีวิตใหม่',
    'ครูไทยเพื่อประชาชน', 'ประชาชาติ', 'พลังธรรมใหม่', 'กรีน',
    'แผ่นดินธรรม', 'ประชาไทย', 'ประชาอาสาชาติ',
    'เครือข่ายชาวนาแห่งประเทศไทย', 'ไทยรวมไทย', 'พลังไทยรักชาติ'
]

def is_party(text, threshold=70):
    match = process.extractOne(text, KNOWN_PARTIES, scorer=fuzz.ratio)
    return match and match[1] >= threshold

In [8]:
def extract_party_score_dict(md: str) -> dict:
    result = {}
    party_idx = None
    vote_idx = None

    for line in md.split('\n'):
        if '---' in line or 'รวมคะแนน' in line:
            continue

        parts = [p.strip() for p in line.split('|') if p.strip()]
        if len(parts) < 2:
            continue

        # Auto-detect column positions from first data row with a known party
        if party_idx is None:
            for i, p in enumerate(parts):
                if is_party(p):
                    party_idx = i
                    # Vote column is always the next one after party
                    vote_idx = i + 1
                    break
            if party_idx is None:
                continue

        if len(parts) <= max(party_idx, vote_idx):
            continue

        party = parts[party_idx].strip()
        vote_raw = parts[vote_idx].strip()

        if not is_party(party):
            continue

        try:
            vote = thai_num_to_int(vote_raw)
            if party:
                result[party] = vote
        except Exception:
            pass

    return result

In [9]:
def extraction(path):
    try:
        if not os.path.exists(path):
            return {}
        filename = os.path.basename(path)
        with open(path, "rb") as f:
            uploaded = client.files.upload(
                file={"file_name": filename, "content": f},
                purpose="ocr"
            )
        signed_url = client.files.get_signed_url(file_id=uploaded.id)
        result = client.ocr.process(
            model="mistral-ocr-latest",
            document={"type": "document_url", "document_url": signed_url.url}
        )
        # Combine all pages markdown
        print(result.pages[0].markdown)
        md = "".join(page.markdown for page in result.pages)
        return extract_party_score_dict(md)
    except Exception as e:
        print(e.__str__())
        return {}

In [17]:
d = extraction('../pdf/constituency_10_11.pdf')
d

|  หมายเลข
ประจําตัวผู้สมัคร | ชื่อ - สกุล
ผู้สมัครรับเลือกตั้ง | สังกัด
พรรคการเมือง | ได้คะแนน
(ให้กรอกทั้งตัวเลขและตัวอักษร)  |
| --- | --- | --- | --- |
|  ๑๕ | นางสาวศศินันท์ ธรรมนิฐินันท์ | ประชาชน | ๓๘,๗๗
(สวมหมื่นแปดพันเจ็ดร้อยเจ็ดส้ม
เก้า)  |
|  หมายเลข
ประจําตัวผู้สมัคร | ชื่อ - สกุล
ผู้สมัครรับเลือกตั้ง | สังกัด
พรรคการเมือง | ได้คะแนน
(ให้กรอกทั้งตัวเลขและตัวอักษร)  |
|  ๗ | นางสาวรัตติกาล แก้วเกิดมี | เพื่อไทย | ๒๔,๘๕๖
(สองหมื่นสี่พันแปดร้อยห้าส้ม)  |
|  ๑๓ | นายเอกภพ เหลืองประเสริฐ | ภูมิใจไทย | ๒๖,๕๔๘
(สองหมื่นสี่ร้อยเก้าส้มแปด)  |
|  ๑๔ | นางสาวรมิดา อินทะแพทย์ | ประชาธิปไตย | ๓,๘๓
(สวมพันแปดร้อยสวมส้มเก้า)  |
|  ๘ | นางสาวธนากา อัครมะหะเวทน์ | ประชากรไทย | ๑,๑๖๘
(หนึ่งพันหนึ่งร้อยแปด)  |
|  ๕ | นายกร สิงห์ธีร์ | รวมไทยสร้างชาติ | ๙๔๖
(เก้าร้อยเก้าส้มหก)  |
|  ๑๖ | นายธนธรรม เจิดรังษี | เศรษฐกิจ | ๙๔๖
(เก้าร้อยเก้าส้มหก)  |
|  ๖ | นายพงศกร ตันติพรหม | ไทยภักดี | ๖๘๖
(หกร้อยแปดส้มหก)  |
|  ๑๑ | ร.ต. เทวิน พิมพ์พันธุ์ | อนาคตไทย | ๖๒๖
(หกร้อยยี่สิบ)  |
|  ๑๗ | ร.ท. จักร์ 

{'ประชาชน': 3877,
 'เพื่อไทย': 24856,
 'ภูมิใจไทย': 26548,
 'ประชาธิปไตย': 383,
 'ประชากรไทย': 1168,
 'รวมไทยสร้างชาติ': 946,
 'เศรษฐกิจ': 946,
 'ไทยภักดี': 686,
 'อนาคตไทย': 626,
 'โอกาสใหม่': 585,
 'ไทยสร้างไทย': 533,
 'บ้านเมือง': 369,
 'ไทยก้าวใหม่': 376,
 'พลังประชารัฐ': 296,
 'กล้าธรรม': 263,
 'พลวัต': 185,
 'ประชาธิปไตยใหม่': 188,
 'วิชชั่นใหม่': 66}